12 .- Implementa descenso de gradiente por lotes con detención temprana para una regresión softmax sin utilitzar SCikit - Learn, solo NumPy. Úsalo en una tarea de classificación como el conjunto de datos IRIS.

# Importar el dataset

In [2]:
import numpy as np

# ============================================================
# 1. Cargar Iris sin Scikit-Learn
# ============================================================

# También puedes descargar iris.data manualmente desde UCI y poner:
# raw = np.genfromtxt("iris.data", delimiter=",", dtype=str)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
raw = np.genfromtxt(url, delimiter=",", dtype=str)

# Eliminamos posibles filas vacías
raw = raw[raw[:, 0] != ""]

# Usamos solo longitud y anchura del pétalo, como en el capítulo
X = raw[:, [2, 3]].astype(float)

# Convertimos etiquetas de texto a números: 0, 1, 2
class_names, y = np.unique(raw[:, 4], return_inverse=True)

print(class_names)
print(X.shape, y.shape)

['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']
(150, 2) (150,)


# Crear conjuntos de test, validacion i entrenamiento

In [3]:
# ============================================================
# 2. Dividir manualmente en train, validación y test
# ============================================================

test_ratio = 0.2
valid_ratio = 0.2

m = len(X)

test_size = int(m * test_ratio)
valid_size = int(m * valid_ratio)
train_size = m - test_size - valid_size

rng = np.random.default_rng(42)
indices = rng.permutation(m)

train_indices = indices[:train_size]
valid_indices = indices[train_size:train_size + valid_size]
test_indices = indices[train_size + valid_size:]

X_train = X[train_indices]
y_train = y[train_indices]

X_valid = X[valid_indices]
y_valid = y[valid_indices]

X_test = X[test_indices]
y_test = y[test_indices]

# Transformación de datos

In [4]:
# ============================================================
# 3. Escalar manualmente usando media y desviación de train
# ============================================================

mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

X_train_scaled = (X_train - mean) / std
X_valid_scaled = (X_valid - mean) / std
X_test_scaled = (X_test - mean) / std

In [5]:
# ============================================================
# 4. Añadir columna de bias: x0 = 1
# ============================================================

X_train_b = np.c_[np.ones(len(X_train_scaled)), X_train_scaled]
X_valid_b = np.c_[np.ones(len(X_valid_scaled)), X_valid_scaled]
X_test_b = np.c_[np.ones(len(X_test_scaled)), X_test_scaled]

In [6]:
# ============================================================
# 5. One-hot encoding
# ============================================================

def to_one_hot(y, n_classes):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1
    return Y

n_classes = len(np.unique(y))

Y_train = to_one_hot(y_train, n_classes)
Y_valid = to_one_hot(y_valid, n_classes)
Y_test = to_one_hot(y_test, n_classes)

# Funciones

In [7]:
# ============================================================
# 6. Función softmax
# ============================================================

def softmax(logits):
    # Restamos el máximo para mejorar la estabilidad numérica
    logits_stable = logits - logits.max(axis=1, keepdims=True)
    exp_scores = np.exp(logits_stable)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

In [8]:
# ============================================================
# 7. Cross-entropy loss + regularización L2
# ============================================================

def compute_loss(X_b, Y, Theta, alpha=0.0):
    epsilon = 1e-15
    
    logits = X_b @ Theta
    Y_proba = softmax(logits)
    
    cross_entropy = -np.mean(
        np.sum(Y * np.log(Y_proba + epsilon), axis=1)
    )
    
    # No regularizamos la primera fila de Theta porque es el bias
    l2_penalty = 0.5 * alpha * np.sum(Theta[1:] ** 2)
    
    return cross_entropy + l2_penalty

# Entrenamiento

In [9]:
# ============================================================
# 8. Batch Gradient Descent con early stopping
# ============================================================

eta = 0.1
alpha = 0.01
n_epochs = 50_000
patience = 100
min_delta = 1e-7

m_train = len(X_train_b)
n_inputs = X_train_b.shape[1]
n_outputs = n_classes

rng = np.random.default_rng(42)
Theta = rng.normal(size=(n_inputs, n_outputs))

best_Theta = Theta.copy()
best_valid_loss = np.inf
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(n_epochs):
    # Forward pass
    logits = X_train_b @ Theta
    Y_proba = softmax(logits)
    
    # Gradiente de cross-entropy
    error = Y_proba - Y_train
    
    gradients = (1 / m_train) * X_train_b.T @ error
    
    # Gradiente de regularización L2
    # No regularizamos el bias
    gradients[1:] += alpha * Theta[1:]
    
    # Actualización de parámetros
    Theta = Theta - eta * gradients
    
    # Calculamos pérdida de validación
    valid_loss = compute_loss(X_valid_b, Y_valid, Theta, alpha)
    
    # Early stopping
    if valid_loss < best_valid_loss - min_delta:
        best_valid_loss = valid_loss
        best_Theta = Theta.copy()
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        
        if epochs_without_improvement >= patience:
            print(f"Early stopping en epoch {epoch}")
            print(f"Mejor epoch: {best_epoch}")
            print(f"Mejor valid loss: {best_valid_loss:.4f}")
            break

Theta = best_Theta

Early stopping en epoch 2052
Mejor epoch: 1952
Mejor valid loss: 0.2710


# Inferencia

In [10]:
# ============================================================
# 9. Predicción
# ============================================================

def predict(X_b, Theta):
    logits = X_b @ Theta
    Y_proba = softmax(logits)
    return Y_proba.argmax(axis=1)

y_train_pred = predict(X_train_b, Theta)
y_valid_pred = predict(X_valid_b, Theta)
y_test_pred = predict(X_test_b, Theta)

train_accuracy = np.mean(y_train_pred == y_train)
valid_accuracy = np.mean(y_valid_pred == y_valid)
test_accuracy = np.mean(y_test_pred == y_test)

print("Train accuracy:", train_accuracy)
print("Valid accuracy:", valid_accuracy)
print("Test accuracy:", test_accuracy)

Train accuracy: 0.9555555555555556
Valid accuracy: 0.9666666666666667
Test accuracy: 0.9666666666666667
